# LTSR — Colab 実行ノートブック

**層選択型 Transformer シンボリック回帰**の D-アクション改善版パイプラインを Colab で実行します。

> **Colab に PowerShell はありません。** Colab は Linux(Ubuntu)+Jupyter で、セル内で `!コマンド`(bash) や
> `%cd` などのマジックを使います。ローカルの PowerShell 手順はそのまま動かないため、このノートブックを使ってください。

### 実行順（D の推奨アクションを反映）
1. 多骨格・構造分割の合成スイート生成（+ノイズ水準）
2. **Phase 4 multi-seed** — 層寄与を複数seedで CI 付き推定（A-1）
3. **Phase 5** — Phase 4 の `contributions.json` から動的にランキング取得（B-1）、random対照は top-k除外（A-3）
4. **Phase 6 noise sweep** — H3 ノイズ頑健性（A-4）
5. **Phase 8 LODO** — donor 交差検証で汎化ギャップ（deep-dive）

### 前提
- **ランタイム**: 上部メニュー『ランタイム > ランタイムのタイプを変更 > GPU (T4等)』を選択。
- **Python 3.10 が必要**（NeSymReS が `hydra-core==1.0.0` に依存し、Python 3.11/3.12 だと import 時に
  `mutable default ... override_dirname` エラーになる）。Colab が 3.11/3.12 の場合は下の condacolab セルで 3.10 環境を作ります。
- **先に GitHub へ push**: このノートは repo を `git clone` します。ローカルで追加した新スクリプト
  (`phase4_multiseed.py` 等) を **push 済み**にしてから実行してください。未 push なら Drive アップロードで代替可(末尾の付録)。

## 0. Python バージョン確認

In [ ]:
import sys
print('Python', sys.version)
major, minor = sys.version_info[:2]
if (major, minor) >= (3, 11):
    print('\n⚠️ Python >= 3.11 です。次の condacolab セルを実行して 3.10 環境を作ってください。')
else:
    print('\n✅ Python 3.10 系。condacolab セルはスキップして構いません。')

## 1. (必要時のみ) condacolab で Python 3.10 環境

上のセルで Python>=3.11 と出た場合だけ実行。**実行するとカーネルが自動再起動**します（正常動作）。
再起動後は **このセルは飛ばして** 次の『2. リポジトリ取得』から続けてください。

In [ ]:
# Python>=3.11 のときだけ実行（カーネルが再起動します）
!pip install -q condacolab
import condacolab
condacolab.install()  # 3.10 の conda 環境を構築し、カーネルを再起動

## 2. リポジトリ取得

公開リポジトリ想定。private の場合は `REPO_URL` を
`https://<GITHUB_TOKEN>@github.com/<user>/<repo>.git` の形にしてください。

In [ ]:
REPO_URL = "https://github.com/blabo25226/Layer-selective_Transformer-based_Symbolic_Regression.git"
BRANCH   = "main"   # D-アクションを別ブランチに push した場合はそのブランチ名に

%cd /content
import os
if not os.path.isdir('/content/LTSR'):
    !git clone --branch $BRANCH $REPO_URL LTSR
%cd /content/LTSR
!git pull --ff-only || true
print('\n== scripts present? ==')
!ls scripts/ | grep -E 'phase4_multiseed|phase6_noise_sweep|phase8_lodo|generate_diverse_suite' || \
  echo '!! 新スクリプトが見つかりません → ローカルで git push してから git pull してください'

## 3. 依存関係のインストール

`requirements/colab.txt`(→base.txt) の固定版 + NeSymReS を editable インストール。数分かかります。

In [ ]:
%cd /content/LTSR
!pip install -q -r requirements/colab.txt
!pip install -q -e NSRS/src
# TPSR アダプタ（MCTS）が使う依存
!pip install -q sympytorch zss 'gym==0.26.2'
print('deps installed')

## 4. NeSymReS 事前学習重みの取得

HuggingFace `TommasoBendinelli/NeuralSymbolicRegressionThatScales` から `10M.ckpt` を取得し
`NSRS/weights/` へ配置。TPSR も同じ重みを共有（symlink）。

In [ ]:
import os, shutil
from huggingface_hub import hf_hub_download
os.makedirs('NSRS/weights', exist_ok=True)
if not os.path.exists('NSRS/weights/10M.ckpt'):
    ck = hf_hub_download(repo_id='TommasoBendinelli/NeuralSymbolicRegressionThatScales',
                         filename='10M.ckpt')
    shutil.copy(ck, 'NSRS/weights/10M.ckpt')
# TPSR は nesymres バックボーンで同じ重みを再利用
os.makedirs('TPSR/nesymres/weights', exist_ok=True)
dst = 'TPSR/nesymres/weights/10M.ckpt'
if not os.path.exists(dst):
    os.symlink(os.path.abspath('NSRS/weights/10M.ckpt'), dst)
print('weights ready:', os.path.getsize('NSRS/weights/10M.ckpt')//10**6, 'MB')

## 5. 動作確認（import & GPU）

ここで `hydra` の mutable default エラーが出たら Python が 3.11/3.12 のままです → 手順1へ戻る。

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
# NeSymReS が import できるか（hydra 互換性の確認）
import sys; sys.path.insert(0, 'NSRS/src')
from nesymres.architectures.model import Model
print('✅ NeSymReS import OK')

In [ ]:
# Phase 0 の環境チェックスクリプト
!python scripts/phase0_check_environment.py || true

## 6. PySR (Julia) セットアップ *(Phase 8 の PySR baseline 用・任意)*

初回 import で Julia バックエンドを自動インストール（数分）。Phase 8 を `--with-pysr` で走らせる場合のみ必要。

In [ ]:
try:
    import pysr
    try:
        pysr.install()  # 旧API。新しい pysr では no-op/不要
    except Exception as e:
        print('pysr.install skipped:', e)
    print('✅ pysr ready')
except Exception as e:
    print('PySR セットアップ失敗（--with-pysr を外せば他は動きます）:', e)

---
# D-アクション パイプライン

各セルは結果を `results/phase_results/` 以下に書き出します。GPU 前提で、所要時間はデータ量/seed 数に依存します（下の既定値で概ね各 10–40 分程度）。

## 7-1. 合成スイート生成（多骨格・構造分割 + ノイズ水準） — A-1 / A-4

10 train / 5 test の**互いに素な骨格**を各 8 パラメータで生成。ノイズ 0/0.05/0.1/0.2 の版も作成。

In [ ]:
!python scripts/generate_diverse_suite.py --n-per-skeleton 8 --noise 0.0 0.05 0.1 0.2
!echo '---'; cat results/synthetic/diverse_v1/suite_report.md

## 7-2. Phase 4 multi-seed（層寄与を CI 付きで） — A-1

3 seed で全条件を学習し、層ごとに寄与 mean±95%CI と top-3 安定度を出力。
CI が重なれば H2(層選択性) は**非支持**と読める。

In [ ]:
!python scripts/phase4_multiseed.py \
    --data-dir results/synthetic/diverse_v1 --seeds 0 1 2 --epochs 3

## 7-3. Phase 5 選択的 FT（動的ランキング + 健全な random 対照） — B-1 / A-3

Phase 4(単一seed)の `contributions.json` からランキングを動的取得。multi-seed の集計を使いたい場合はそちらを別途 contributions.json 化して `--contributions` で渡す。

In [ ]:
!python scripts/phase5_selective_train.py \
    --data-dir results/synthetic/diverse_v1 --epochs 5

## 7-4. Phase 6 ノイズスイープ（H3） — A-4

ノイズ水準 × 2×2(FT×decode) で NMSE/R² の劣化スロープを比較し、H3 を判定。

In [ ]:
!python scripts/phase6_noise_sweep.py \
    --noise 0.0 0.05 0.1 0.2 --epochs 5 --rollout 3 --width 3

## 7-5. Phase 8 LODO（donor 交差検証の汎化ギャップ） — deep-dive

leave-one-donor-out で PySR vs selective-FT の holdout NMSE を集計。PySR を含めると時間がかかるため、まず `--with-pysr` 無しで動作確認しても良い。

In [ ]:
# まず NeSymReS 系のみ（速い）
!python scripts/phase8_lodo.py --epochs 5

In [ ]:
# PySR baseline を含める（Julia 必須・低速）
!python scripts/phase8_lodo.py --epochs 5 --with-pysr --pysr-iters 12

## 8. レポート表示

In [ ]:
import os
from IPython.display import Markdown, display
reports = [
    'phase4_multiseed_report.md',
    'phase5_report.md',
    'phase6_noise_report.md',
    'phase8_lodo_report.md',
]
for r in reports:
    p = f'results/phase_results/{r}'
    if os.path.exists(p):
        display(Markdown(f'# 📄 {r}\n\n' + open(p, encoding='utf-8').read()))
    else:
        print('(missing)', p)

## 9. 結果を Google Drive へ保存

Colab のセッションは切断で消えるため、`results/` を Drive にコピーしておく。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import time, os
dst = f"/content/drive/MyDrive/LTSR_results_{time.strftime('%Y%m%d_%H%M')}"
os.makedirs(dst, exist_ok=True)
!cp -r results "$dst"/
print('saved to', dst)

---
## 付録A: private リポジトリ / push していない場合（Drive から実行）

GitHub に push せず、ローカルの repo を Google Drive に置いて使う場合：
1. ローカルの `LTSR` フォルダ一式を Google Drive の `MyDrive/LTSR` にアップロード（重み `NSRS/weights/10M.ckpt` は大きいので手順4で取得推奨、除外可）。
2. 下を実行して clone の代わりに Drive を使う。

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')
# %cd /content/drive/MyDrive/LTSR
# 以降は手順3(依存)・手順4(重み)・手順7(パイプライン)を同様に実行

## 付録B: トラブルシュート

| 症状 | 対処 |
|------|------|
| `ValueError: mutable default ... override_dirname` | Python が 3.11/3.12。手順1の condacolab で 3.10 化 |
| 新スクリプトが無い | ローカルで `git add/commit/push` → Colab で `git pull` |
| `10M.ckpt` が無い/LFS 未取得 | 手順4の HuggingFace ダウンロードを使用（LFS 不要） |
| private repo で clone 失敗 | `REPO_URL` を `https://<TOKEN>@github.com/...` に |
| CUDA out of memory | `--batch-size` を下げる / `--seeds 0 1` に減らす |
| PySR/Julia が遅い・失敗 | Phase 8 を `--with-pysr` 無しで実行 |